# EvalForge — reproducible GPU experiment
Enable internet and a CUDA GPU. This notebook creates a Python 3.11 environment independently of Kaggle’s host Python. The data mirror is pinned and checksum verified; model and dataset terms apply. No secret tokens are required for the public model and data. Outputs contain the adapter, manifest, and paired evaluation report.
Before running: select **GPU T4 x2**, enable Internet, and set Persistence to **Files** (or **Variables & Files** if offered). Use **Save Version → Save & Run All** for the final run so completed output files are saved with the version. After it succeeds, download and verify the ZIP before deleting or resetting anything.


In [ ]:
import subprocess, sys, os
from pathlib import Path
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
subprocess.run(['uv', 'venv', '--python', '3.11', '--seed', '/kaggle/temp/evalforge-env'], check=True)
repo = Path('/kaggle/working/EvalForge')
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Navaneeth-007/EvalForge.git', str(repo)], check=True)
print(subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True))
os.environ['HF_HOME'] = '/kaggle/temp/evalforge-model-cache'
os.environ['CUDA_VISIBLE_DEVICES'] = '0,1'
os.environ['WANDB_MODE'] = 'disabled'
subprocess.run(['uv', 'pip', 'install', '--python', '/kaggle/temp/evalforge-env/bin/python', '-r', str(repo / 'requirements-ml.txt')], check=True)


## Train and evaluate
Full training split, two epochs, two GPUs, and 200 development questions. Distributed training uses batch size 1 per GPU and gradient accumulation 8: effective batch size remains 16. Data preparation and paired evaluation each run once. Start with a short smoke run in a separate notebook if desired; no two-GPU hardware verification has been performed by the maintainer.


In [ ]:
RUN_ARGS = ['--gpus', '2', '--epochs', '2', '--eval-n', '200']
subprocess.run(['/kaggle/temp/evalforge-env/bin/python', '-u', '-m', 'scripts.run_gpu', *RUN_ARGS], cwd=str(repo), env=os.environ.copy(), check=True)
import shutil
shutil.copy2(repo / 'data/processed/eval_samples.json', repo / 'outputs/eval_samples.json')
archive = shutil.make_archive('/kaggle/working/evalforge-evidence', 'zip', repo / 'outputs')
print('DOWNLOAD AND VERIFY:', archive)


## Inspect and save evidence
These are development metrics, not a blind test score. Inspect incorrect predictions before claiming an improvement. Do not publish the model card until its pending fields have been replaced with measured values.


In [ ]:
import json, shutil
manifest = json.loads((repo / 'outputs/phi3-sql-qlora/training_manifest.json').read_text())
report = json.loads((repo / 'outputs/eval_results.json').read_text())
print('Training examples:', manifest['train_examples'])
print('GPU:', manifest['gpu'], 'Elapsed seconds:', manifest['elapsed_seconds'])
print('Base metrics:', report['base_model_metrics'])
print('Adapter metrics:', report['finetuned_model_metrics'])
import hashlib, zipfile
archive_path = Path('/kaggle/working/evalforge-evidence.zip')
with zipfile.ZipFile(archive_path) as z:
    assert z.testzip() is None, 'Corrupt archive'
    assert any(name.endswith('adapter_model.safetensors') for name in z.namelist())
print('Archive bytes:', archive_path.stat().st_size)
print('SHA256:', hashlib.file_digest(archive_path.open('rb'), 'sha256').hexdigest())
